In [2]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import openpyxl
import statsmodels.api as sm

# TREYNOR-BLACK MODEL OVERVIEW (Academic Context)
# Developed by Jack Treynor (1969) & Fischer Black (1972), this model optimally blends
# an active portfolio (mispriced securities) with passive market index.
# Key insight: Use alpha/sigma_e (information ratio) to tilt from market equilibrium.
# Assumptions: CAPM holds (alpha=0 expected), but analyst forecasts nonzero alphas.
# Goal: Portfolio beta=1, max Sharpe via active alpha boost minus nonsystematic risk.
# Formulas from Bodie/Kane/Marcus (Investments): Eq 8.13-8.17.


In [3]:
# =============================================================================
# 1. DATA LOADING AND PREP
# =============================================================================
# Load price data. Returns computed monthly % (finance std).
all_prices = pd.read_excel('All_Raw_Data.xlsx', index_col=0)
all_returns = all_prices.pct_change().dropna() * 100

In [4]:
# =============================================================================
# 2. KEY INPUT PARAMETERS
# =============================================================================
rf_monthly_pct = 0.37 / 100  # Monthly rf (your input).
benchmark_col = 'ACWI.O'     # Passive benchmark (MSCI ACWI proxy).
securities = [c for c in all_returns.columns if c != benchmark_col]  # Active universe.
rf_annual_pct = (1 + rf_monthly_pct)**12 - 1
portfolio_value = 50000000   # £50m fund for position sizing.

In [5]:
# =============================================================================
# 3. CAPM REGRESSIONS: ALPHA, BETA, SIGMA_E EXTRACTION
# Single-index model: r_i = alpha_i + beta_i (r_M - rf) + e_i
# alpha_i: mispricing (forecast edge). sigma_ei: diversifiable firm-specific risk.
# =============================================================================
results = {}
benchmark_ret = all_returns[benchmark_col]
for ticker in securities:
    asset_ret = all_returns[ticker]
    valid = ~(asset_ret.isna() | benchmark_ret.isna())
    if valid.sum() < 12:
        results[ticker] = {'alpha_monthly_pct': 0.0, 'beta': 1.0, 'sigmae_monthly_pct': 1.0}
        continue
    
    y = asset_ret[valid] - rf_monthly_pct * 100  # Excess return.
    x = benchmark_ret[valid] - rf_monthly_pct * 100
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    
    alpha_monthly = float(model.params.iloc[0])
    beta = float(model.params.iloc[1])
    sigmae_monthly = np.sqrt(float(model.mse_resid))
    
    results[ticker] = {'alpha_monthly_pct': alpha_monthly, 'beta': beta, 'sigmae_monthly_pct': sigmae_monthly}

In [6]:
# =============================================================================
# 4. ANNUALISATION (TB Uses Annual Stats)
# Alpha: geometric compound. sigma_e: sqrt(12) scale (uncorrelated residuals).
# =============================================================================
for t in results:
    r = results[t]
    r['alpha_annual_pct'] = (1 + r['alpha_monthly_pct']/100)**12 - 1
    r['sigmae_annual_pct'] = r['sigmae_monthly_pct'] * np.sqrt(12)

# Market: E(r_M), sigma_M for RPM = E(r_M - rf), var_M.
ER_M_monthly = all_returns[benchmark_col].mean()
sigma_M_monthly = all_returns[benchmark_col].std()
ER_M_annual_pct = (1 + ER_M_monthly/100)**12 - 1
sigma_M_annual_pct = sigma_M_monthly * np.sqrt(12)
var_M_annual = (sigma_M_annual_pct / 100)**2
RPM_annual = ER_M_annual_pct - rf_annual_pct

print("Top 5 annual alphas")
for t in sorted(results, key=lambda x: results[x]['alpha_annual_pct'], reverse=True)[:5]:
    r = results[t]
    print(f"{t:15} alpha {r['alpha_annual_pct']:.2f}, sigmae_ann {r['sigmae_annual_pct']:.2f}")

Top 5 annual alphas
000660.KS       alpha 1.29, sigmae_ann 56.54
FUTU.OQ         alpha 0.77, sigmae_ann 73.19
UBER.N          alpha 0.42, sigmae_ann 37.99
6723.T          alpha 0.39, sigmae_ann 44.30
AM.PA           alpha 0.39, sigmae_ann 29.10


In [7]:
# =============================================================================
# 5. ACTIVE PORTFOLIO (Eq 8.21-8.22): w*_i = alpha_i / sigma^2_ei
# Proportional to squared appraisal ratio (alpha/sigma_e). Positive alpha only.
# Normalise to sum=1. Diversifies sigma_eA.
# =============================================================================
w0i = {}
for t in securities:
    if results[t]['alpha_annual_pct'] > 0:
        w0i[t] = (results[t]['alpha_annual_pct']/100) / (results[t]['sigmae_annual_pct']/100)**2

sum_w0i = sum(w0i.values())
wAi = {t: w0i.get(t, 0) / sum_w0i for t in securities} if sum_w0i > 0 else {t: 1/len(securities) for t in securities}

print("\nActive PF weights sum=100% (Eq 8.22)")
for t in sorted(wAi, key=wAi.get, reverse=True)[:10]:
    print(f"{t:15} {wAi[t]*100:.1f}")


Active PF weights sum=100% (Eq 8.22)
AM.PA           11.9
000660.KS       10.6
MEXS LN Equity  10.4
5108.T          10.4
BLK.N           9.2
UTOS.SI         7.6
UBER.N          7.6
MSFT.O          7.1
6723.T          5.3
FLBR US Equity  3.8


In [8]:
# =============================================================================
# 6. ACTIVE AGGREGATES (Eq 8.23)
# Treats active PF as "one security": alpha_A, beta_A, sigma_eA.
# sigma_eA << individual sigma_e (diversification benefit).
# =============================================================================
alpha_A_dec = sum(wAi[t] * (results[t]['alpha_annual_pct']/100) for t in securities)
sigmae_A_dec = np.sqrt(sum((wAi[t]**2 * (results[t]['sigmae_annual_pct']/100)**2) for t in securities))
beta_A = sum(wAi[t] * results[t]['beta'] for t in securities)
alpha_A_pct = alpha_A_dec * 100
sigmae_A_pct = sigmae_A_dec * 100

print(f"\nActive PF (as 'security'): alpha_A {alpha_A_pct:.2f}, sigma_eA {sigmae_A_pct:.2f}, beta_A {beta_A:.2f}")


Active PF (as 'security'): alpha_A 0.34, sigma_eA 9.50, beta_A -0.30


In [9]:
# =============================================================================
# 7. OPTIMAL MIX (Eq 8.24-8.25): ACTIVE vs PASSIVE
# w0: unadjusted weight (like tangency for active).
# wA: beta-adjusted to ensure overall beta=1 (no systematic tilt).
# wM = 1 - wA (pure passive benchmark).
# =============================================================================
w0 = (alpha_A_dec / sigmae_A_dec**2) / ((RPM_annual / 100) / var_M_annual) if sigmae_A_dec > 0 else 0
wA = w0 / (1 + (1 - beta_A) * w0) if abs(1 + (1 - beta_A) * w0) > 1e-6 else 0
wM = 1 - wA  # Implicit passive allocation (not explicitly held).

print(f"\nTB Optimal Mix (Eq 8.24-25)")
print(f"w0 (unadj): {w0:.4f}")
print(f"wA (active): {wA:.4f}")
print(f"wM (passive): {wM:.4f}")
print(f"Overall beta: {wA * beta_A + wM * 1:.2f} (targets 1)")


TB Optimal Mix (Eq 8.24-25)
w0 (unadj): 16.6325
wA (active): 0.7374
wM (passive): 0.2626
Overall beta: 0.04 (targets 1)


In [10]:
# =============================================================================
# 8. PURE TB SECURITY WEIGHTS
# Final: w_i = wA * wAi for active securities only. No explicit ACWI.O holding.
# Passive wM embedded (replication via cash + futures or index).
# =============================================================================
tb_weights = {t: wA * wAi[t] for t in securities}  # Active only!

print("\nPure TB Weights (active securities only, sum=wA)")
for t in sorted(tb_weights, key=lambda k: abs(tb_weights[k]), reverse=True)[:15]:
    print(f"{t:15} {tb_weights[t]*100:.2f}")


Pure TB Weights (active securities only, sum=wA)
AM.PA           8.79
000660.KS       7.81
MEXS LN Equity  7.67
5108.T          7.67
BLK.N           6.79
UTOS.SI         5.63
UBER.N          5.58
MSFT.O          5.22
6723.T          3.89
FLBR US Equity  2.83
FUTU.OQ         2.79
BBRM.NS         2.63
CVX.N           2.23
WISEa.L         2.14
BABA.N          2.07


In [11]:
# =============================================================================
# 9. CONSTRAINTS (Practical Overrides for ICM340 £50m Fund)
# Post-TB: clip positions, cap gearing (gross/net <=150%), min long 1%.
# Rescale preserves ratios while complying.
# =============================================================================
max_pos_pct = 0.25
max_gearing = 1.50
use_min = True

final_weights = tb_weights.copy()  # Start from pure TB active.

# Clip individual positions.
for t in final_weights:
    final_weights[t] = np.clip(final_weights[t], -max_pos_pct, max_pos_pct)

# Iterative shrink if gearing breached (gross exp / net exp).
total_w = sum(final_weights.values())
grossexp = sum(abs(w) for w in final_weights.values())
while grossexp > max_gearing * abs(total_w) and abs(total_w) > 1e-6:
    scale = 0.95
    for t in final_weights:
        final_weights[t] *= scale
    total_w = sum(final_weights.values())
    grossexp = sum(abs(w) for w in final_weights.values())

# Normalise to sum=1 (full allocation).
if abs(total_w) > 1e-6:
    for t in final_weights:
        final_weights[t] /= total_w

print(f"\nConstrained (sum=100%, active only)")
print(f"Gearing gross/net {grossexp:.1f} / 100.0  <=150")
print(f"Max position {max(abs(v) for v in final_weights.values())*100:.1f} <=25")
print("\n" + "-"*50)
print("Asset".ljust(20) + "Weight".ljust(7) + "Value £50m".rjust(15))
print("-"*50)
for t in sorted(final_weights, key=lambda x: abs(final_weights[x]), reverse=True)[:20]:
    w = final_weights[t]
    print(f"{t}".ljust(20) + f"{w*100:6.1f}" + f"{w * portfolio_value:15,.0f}")


Constrained (sum=100%, active only)
Gearing gross/net 0.7 / 100.0  <=150
Max position 11.9 <=25

--------------------------------------------------
Asset               Weight      Value £50m
--------------------------------------------------
AM.PA                 11.9      5,961,432
000660.KS             10.6      5,295,762
MEXS LN Equity        10.4      5,203,779
5108.T                10.4      5,199,287
BLK.N                  9.2      4,605,139
UTOS.SI                7.6      3,815,790
UBER.N                 7.6      3,781,841
MSFT.O                 7.1      3,541,321
6723.T                 5.3      2,635,345
FLBR US Equity         3.8      1,919,753
FUTU.OQ                3.8      1,889,385
BBRM.NS                3.6      1,785,198
CVX.N                  3.0      1,509,236
WISEa.L                2.9      1,453,449
BABA.N                 2.8      1,403,283
SASY.PA                0.0              0
VITP                   0.0              0
IEF.O                  0.0              0
I

In [12]:
# =============================================================================
# 10. VALIDATION METRICS
# Overall beta ≈1 (systematic match). TE: undiversifiable nonsystematic risk.
# M2/Sharpe superior if alphas valid (TB promise).
# =============================================================================
final_beta = sum(final_weights[t] * results[t]['beta'] for t in securities)
final_te_pct = np.sqrt(sum((final_weights[t]**2 * (results[t]['sigmae_annual_pct']/100)**2) for t in securities)) * 100
print(f"\nFinal beta {final_beta:.2f} (pure TB=1; constraints tilt slightly)")
print(f"Final TE (nonsystematic risk) {final_te_pct:.1f}%")


Final beta -0.30 (pure TB=1; constraints tilt slightly)
Final TE (nonsystematic risk) 9.5%
